# Movie Review Sentiment Analyzer
**PCCSL 508 – Machine Learning Lab | Microproject**

- **Dataset:** IMDb Reviews (50,000 reviews: 25k train / 25k test)
- **Model:** DistilBERT (fine-tuning)
- **Metrics:** Accuracy, Precision, Recall, F1, Confusion Matrix
- **Deployment:** Gradio

> **Before running:** Runtime → Change runtime type → **T4 GPU**

## 1. Install libraries

In [ ]:
!pip install -q transformers datasets evaluate accelerate gradio scikit-learn

## 2. Imports and GPU check

In [ ]:
import re
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns

from datasets import load_dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding,
                          pipeline)
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             confusion_matrix, classification_report)

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

## 3. Load the IMDb dataset

In [ ]:
dataset = load_dataset("imdb")
print(dataset)
print("\nSample review:\n", dataset["train"][0]["text"][:300])
print("Label (0 = negative, 1 = positive):", dataset["train"][0]["label"])

## 4. Exploratory analysis

In [ ]:
df = pd.DataFrame(dataset["train"])
print("Class distribution:")
print(df["label"].value_counts())

df["word_count"] = df["text"].apply(lambda x: len(x.split()))
print("\nAverage words per review:", round(df["word_count"].mean(), 1))

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(x="label", data=df, ax=ax[0])
ax[0].set_title("Class Distribution (0=Neg, 1=Pos)")
ax[1].hist(df["word_count"], bins=50, color="steelblue")
ax[1].set_title("Review Length (words)")
ax[1].set_xlabel("Words")
plt.tight_layout()
plt.show()

## 5. Preprocessing
Only light cleaning is needed: remove HTML tags (`<br />`) and extra spaces.
We do **not** remove stopwords or lowercase manually, because DistilBERT's tokenizer handles that and needs full sentences for context.

In [ ]:
def clean_text(example):
    text = example["text"]
    text = re.sub(r"<.*?>", " ", text)      # remove HTML tags
    text = re.sub(r"\s+", " ", text).strip()  # remove extra whitespace
    example["text"] = text
    return example

dataset = dataset.map(clean_text)

# Create a validation set (10% of training data)
split = dataset["train"].train_test_split(test_size=0.1, seed=42, stratify_by_column="label")
train_ds = split["train"]
val_ds   = split["test"]
test_ds  = dataset["test"]

print("Train:", len(train_ds), "| Validation:", len(val_ds), "| Test:", len(test_ds))

## 6. Tokenization

In [ ]:
MODEL_NAME = "distilbert-base-uncased"
MAX_LEN = 256

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LEN)

train_tok = train_ds.map(tokenize, batched=True)
val_tok   = val_ds.map(tokenize, batched=True)
test_tok  = test_ds.map(tokenize, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

## 7. Load pre-trained DistilBERT

In [ ]:
id2label = {0: "NEGATIVE", 1: "POSITIVE"}
label2id = {"NEGATIVE": 0, "POSITIVE": 1}

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2, id2label=id2label, label2id=label2id
)

## 8. Training setup

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "precision": precision, "recall": recall, "f1": f1}

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=2,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=100,
    fp16=torch.cuda.is_available(),
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

## 9. Train the model (about 10–15 minutes on a T4 GPU)

In [ ]:
trainer.train()

## 10. Evaluate on the test set

In [ ]:
results = trainer.evaluate(test_tok)
print("Test results:")
for k, v in results.items():
    if k.startswith("eval_") and k not in ("eval_loss", "eval_runtime", "eval_samples_per_second", "eval_steps_per_second"):
        print(f"  {k[5:].capitalize():10s}: {v:.4f}")

## 11. Confusion matrix and classification report

In [ ]:
pred_out = trainer.predict(test_tok)
y_pred = np.argmax(pred_out.predictions, axis=-1)
y_true = pred_out.label_ids

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Negative", "Positive"], yticklabels=["Negative", "Positive"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix - IMDb Test Set")
plt.show()

print(classification_report(y_true, y_pred, target_names=["Negative", "Positive"], digits=4))

## 12. Save the model

In [ ]:
SAVE_DIR = "./sentiment_model"
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

# Zip it so you can download it
!zip -r sentiment_model.zip sentiment_model
print("Saved. Download 'sentiment_model.zip' from the Files panel on the left.")

# OPTIONAL: save to Google Drive instead
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r sentiment_model /content/drive/MyDrive/

## 13. Quick test with custom reviews

In [ ]:
clf = pipeline("text-classification", model=SAVE_DIR, tokenizer=SAVE_DIR,
               device=0 if torch.cuda.is_available() else -1)

samples = [
    "An absolute masterpiece! The acting and story were brilliant.",
    "Terrible movie. Boring plot and awful acting. Waste of time.",
    "It was okay, some good moments but a bit too long.",
]
for s in samples:
    r = clf(s)[0]
    print(f"{r['label']:9s} ({r['score']:.2%})  ->  {s}")

## 14. Gradio web app (deployment)

In [ ]:
import gradio as gr

def predict_sentiment(review):
    if not review or not review.strip():
        return {"POSITIVE": 0.0, "NEGATIVE": 0.0}
    review = re.sub(r"<.*?>", " ", review)
    out = clf(review, top_k=None, truncation=True, max_length=MAX_LEN)
    return {o["label"]: float(o["score"]) for o in out}

demo = gr.Interface(
    fn=predict_sentiment,
    inputs=gr.Textbox(lines=6, placeholder="Type or paste a movie review here...", label="Movie Review"),
    outputs=gr.Label(num_top_classes=2, label="Predicted Sentiment"),
    title="Movie Review Sentiment Analyzer",
    description="Fine-tuned DistilBERT on the IMDb dataset. Enter a review to see if it is positive or negative.",
    examples=[
        ["This film was a stunning achievement with brilliant performances."],
        ["I regret watching this. The script was dull and the ending made no sense."],
    ],
)
demo.launch(share=True)  # gives a public link you can show in your viva